# Module 4: Clean FL Baselines


## Stage Goal

Run the clean FL baselines once and save `artifacts/module4_clean_baselines.json`. This notebook keeps training output visible so you can watch each clean algorithm progress. Run `train_v3.ipynb` first so the MobileNetV3 target checkpoint exists.


## 1. Notebook Setup


In [1]:
from pathlib import Path
import sys

MODULE_DIR = Path.cwd()
if (MODULE_DIR / "4_Adversarial_FL").exists():
    MODULE_DIR = MODULE_DIR / "4_Adversarial_FL"
SRC_DIR = MODULE_DIR / "src"
for path in (MODULE_DIR.parent, SRC_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from IPython.display import display
import matplotlib.pyplot as plt

from attack_notebook_utils import (
    clean_result_from_artifact,
    plot_clean_baselines_summary,
    plot_clean_history,
    prepare_context,
    run_clean_baselines,
)

plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})


## 2. Configuration

Edit this cell to change data, FL, or clean-baseline settings. No YAML config is used.


In [4]:
BASE_FED_CONFIG = {
    "fraction_clients": 0.2,
    "num_clients": 100,
    "num_rounds": 10,
    "num_epochs": 5,
    "batch_size": 64,
    "global_stepsize": 1.0,
    "local_stepsize": 0.002,
    "criterion": "torch.nn.CrossEntropyLoss",
}

ALGORITHMS = {
    "FedAvg": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 1.0},
        "optim_config": {},
    },
    "FedAdam": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.01},
        "optim_config": {"beta1": 0.9, "beta2": 0.99, "epsilon": 1e-2},
    },
    "FedAdagrad": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.01},
        "optim_config": {"epsilon": 1e-2},
    },
    "FedYogi": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.01},
        "optim_config": {"beta1": 0.9, "beta2": 0.99, "epsilon": 1e-2},
    },
    "Scaffold": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 1.0},
        "optim_config": {"c_init": 0.0},
    },
}

CONFIG = {
    "quiet": False,
    "clean_baseline_algorithms": ["FedAvg", "FedAdam", "FedAdagrad", "FedYogi", "Scaffold"],
    "data_config": {
        "dataset_path": "./Data/Imagenette",
        "dataset_name": "Imagenette",
        "non_iid_per": 0,
        "eval_subset": "attack_eval",
        "validation_split": {"enabled": True, "seed": 42, "selection_fraction": 0.5},
    },
    "global_config": {"seed": 42, "device": "cuda"},
    "artifacts": {
        "dir": "artifacts",
        "target_checkpoint": "module4_v3_target.pt",
        "surrogate_checkpoint": "module4_surrogate.pt",
        "clean_baselines": "module4_clean_baselines.json",
    },
    "model_config": {
        "module": "model",
        "name": "MobileNetV3Transfer",
        "kwargs": {"pretrained": False, "num_classes": 10, "dropout": 0.1},
    },
    "algorithms": ALGORITHMS,
    "attack": {
        "seed": 42,
        "malicious_fraction": 0.0,
        "malicious_client_selection": {"mode": "none", "client_ids": []},
        "start_round": 3,
        "attack": {
            "type": "random_noise",
            "target_label": 0,
            "poison_rate": 0.0,
            "step_size": 0.0,
            "criterion": "torch.nn.CrossEntropyLoss",
        },
        "surrogate": {
            "checkpoint": "artifacts/module4_surrogate.pt",
            "pretrained": False,
            "num_classes": 10,
            "finetune_epochs": 0,
        },
    },
}

context = prepare_context(CONFIG)
print(f"Device: {context['global_config']['device']}")
print(f"Target checkpoint: {context['target_checkpoint']}")
print(f"Clean baseline artifact: {context['artifact_dir'] / CONFIG['artifacts']['clean_baselines']}")


Device: cuda
Target checkpoint: /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_v3_target.pt
Clean baseline artifact: /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_clean_baselines.json


## 3. Run Clean Baselines

This runs clean FL for the algorithms used by the attack notebooks and writes one shared artifact. Round-by-round training output is shown below.


In [5]:
clean_baselines = run_clean_baselines(
    context,
    algorithms=CONFIG["clean_baseline_algorithms"],
)

display(clean_baselines["table"])
print(f"Saved clean baselines to {clean_baselines['path']}")


Clients successfully initialised

Communication Round: 1



Preparing data with Dirichlet partitioner (aligned with Module 2)
Loading cached client data from cache/client_data_9b5ab701ecf4bad27b1ab2a8e0a01a5f.pkl
[Server] Round 1/10 → selected 20 clients, 5 local epoch(s)


	client_update completed
	server_update completed
	Loss: 0.0767   Accuracy: 97.66%

Communication Round: 2


	Server Loss: 0.0767   Accuracy: 97.66%
[Server] Round 2/10 → selected 20 clients, 5 local epoch(s)


	client_update completed
	server_update completed
	Loss: 0.0768   Accuracy: 97.66%

Communication Round: 3


...

	Server Loss: 0.0770   Accuracy: 97.66%
[ScaffoldServer] Round 10/10 → selected 20 clients, 5 local epoch(s)


	client_update completed
	server_update completed
	Loss: 0.0770   Accuracy: 97.66%


	Server Loss: 0.0770   Accuracy: 97.66%


,algorithm,final_accuracy,final_loss,global_target_label_asr,num_rounds
0,FedAvg,97.71,0.0770,0.23,10
1,FedAdam,97.71,0.0770,0.23,10
2,FedAdagrad,97.66,0.0772,0.23,10
3,FedYogi,97.71,0.0770,0.23,10
4,Scaffold,97.66,0.0770,0.23,10


Saved clean baselines to /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_clean_baselines.json


## 4. Final Clean Baseline Plots


In [6]:
display(clean_baselines["table"])
plot_clean_baselines_summary(clean_baselines, title="Clean FL baselines")
plot_clean_history(
    clean_result_from_artifact(clean_baselines, "FedAvg"),
    title="Clean FedAvg training curve",
)
print("Attack notebooks read this artifact:")
print(clean_baselines["path"])


,algorithm,final_accuracy,final_loss,global_target_label_asr,num_rounds
0,FedAvg,97.71,0.0770,0.23,10
1,FedAdam,97.71,0.0770,0.23,10
2,FedAdagrad,97.66,0.0772,0.23,10
3,FedYogi,97.71,0.0770,0.23,10
4,Scaffold,97.66,0.0770,0.23,10


Attack notebooks read this artifact:
/home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_clean_baselines.json
